In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pathlib import Path
ROOT      = Path().resolve()
import sys
sys.path.insert(0, str(ROOT / 'src' / '2_gold'))
DATA_PATH = ROOT / "data" / "processed"

In [2]:
from cap3_cs_features import gerar_features
from cap3_survival_features import (
    carregar_receitas,
    calc_cumulative,
    JANELAS, DURACAO, FONTES_PARTIDO,
)

In [3]:
rrd = pd.read_parquet(DATA_PATH / "rrd_df_novo.parquet")
rrd["nr_candidato"] = rrd["nr_candidato"].astype(str)

ELEITO_CATS = ["ELEITO", "ELEITO POR MEDIA", "MEDIA", "ELEITO POR QP"]
rrd["eleito"] = rrd["ds_sit_tot_turno"].isin(ELEITO_CATS).astype(int)

rrd_df_merge = gerar_features(rrd)

# exp_deputado_federal: usado nas análises por experiência (não computado por gerar_features)
rrd_df_merge["exp_deputado_federal"] = pd.cut(
    rrd_df_merge["n_eleicoes_deputado_federal"].fillna(0).astype(int),
    bins=[-0.1, 0, 1, 100],
    labels=["Sem experiencia", "1 mandato", "2+ mandatos"],
)

print("candidato_competitivo:", rrd_df_merge["candidato_competitivo"].value_counts().to_dict())
print("por ano:")
print(rrd_df_merge.groupby("ano_eleicao")["candidato_competitivo"].value_counts().unstack())

candidato_competitivo: {False: 20690, True: 2793}
por ano:
candidato_competitivo  False  True 
ano_eleicao                        
2014                    5558    620
2018                    6744    886
2022                    8388   1287


In [4]:
# Dados transacionais — todos os tipos de receita (leitura pesada, ~1-2 min)
df_rec = carregar_receitas()
df_rec_partido = df_rec[df_rec["fonte_tipo"].isin(FONTES_PARTIDO)].copy()

# Dias e semana dentro da janela de campanha
for _df in [df_rec, df_rec_partido]:
    _df["dias_campanha"] = _df.apply(
        lambda r: (r["dt_receita"] - JANELAS[r["ano_eleicao"]][0]).days, axis=1
    )
    _df["semana"] = (_df["dias_campanha"] // 7) + 1

print(f"Receita total — TODOS os tipos (R$ milhões):")
print((df_rec.groupby("ano_eleicao")["vr_receita"].sum() / 1e6).round(1))

Receita total — TODOS os tipos (R$ milhões):
ano_eleicao
2014    1035.4
2018    1298.1
2022    3171.7
Name: vr_receita, dtype: float64


In [5]:
# Vincular experiência como dep. federal
exp_cols = ["ano_eleicao", "sg_uf", "sg_partido", "nr_candidato",
            "n_eleicoes_deputado_federal", "exp_deputado_federal"]
_exp_lookup = rrd_df_merge[exp_cols].drop_duplicates()

for _df in [df_rec, df_rec_partido]:
    _df.drop(columns=["n_eleicoes_deputado_federal", "exp_deputado_federal"],
             errors="ignore", inplace=True)
    _df.merge(_exp_lookup, on=["ano_eleicao", "sg_uf", "sg_partido", "nr_candidato"],
              how="left").pipe(lambda x: _df.update(x))

df_rec = df_rec.merge(
    _exp_lookup, on=["ano_eleicao", "sg_uf", "sg_partido", "nr_candidato"], how="left"
)
df_rec_partido = df_rec_partido.merge(
    _exp_lookup, on=["ano_eleicao", "sg_uf", "sg_partido", "nr_candidato"], how="left"
)

# Cálculos de fluxo acumulado agregados por experiência
df_cum_overall = calc_cumulative(df_rec_partido, ["ano_eleicao"])
df_cum_exp     = calc_cumulative(df_rec_partido, ["ano_eleicao", "exp_deputado_federal"])

print("Dados prontos.")

Dados prontos.


## Fluxo Cumulativo por Experiência

In [6]:
# Tabela-resumo: marcos acumulados por grupo de experiência e ano
# † 2014 = FP + PJ_ROTEADO (pré-FEFC); 2018/2022 = FEFC + FP

cores  = {"Sem experiencia": "lightgrey", "1 mandato": "steelblue", "2+ mandatos": "black"}
tracas = {"Sem experiencia": "dot",       "1 mandato": "dash",       "2+ mandatos": "solid"}
grupos = ["Sem experiencia", "1 mandato", "2+ mandatos"]

rows = []
for ano in [2014, 2018, 2022]:
    fonte_nota = " †" if ano == 2014 else ""
    for grupo in grupos:
        d = df_cum_exp[
            (df_cum_exp["ano_eleicao"] == ano) & (df_cum_exp["exp_deputado_federal"] == grupo)
        ].sort_values("semana")

        total_milhoes = d["vr_receita"].sum() / 1e6

        def pct_ate_semana(s):
            linha = d[d["semana"] == s]
            return linha["cum_prop"].values[0] if len(linha) else np.nan

        mediana = d[d["cum_prop"] >= 0.5]["semana"].min()
        rows.append({
            "Ano": f"{ano}{fonte_nota}", "Grupo": grupo,
            "Total (R$ mi)": round(total_milhoes, 1),
            "% até S1": f"{pct_ate_semana(1):.1%}",
            "% até S2": f"{pct_ate_semana(2):.1%}",
            "% até S4": f"{pct_ate_semana(4):.1%}",
            "Semana mediana (50%)": int(mediana) if not pd.isna(mediana) else "—",
        })

print("† 2014 = FP + PJ_ROTEADO (pré-FEFC); 2018/2022 = FEFC + FP")
pd.DataFrame(rows)

† 2014 = FP + PJ_ROTEADO (pré-FEFC); 2018/2022 = FEFC + FP


,Ano,Grupo,Total (R$ mi),% até S1,% até S2,% até S4,Semana mediana (50%)
0,2014 †,Sem experiencia,28.6,0.1%,14.6%,17.8%,9
1,2014 †,1 mandato,10.2,0.0%,20.6%,24.1%,9
2,2014 †,2+ mandatos,15.8,0.0%,5.9%,9.2%,9
3,2018,Sem experiencia,586.1,8.1%,41.3%,69.5%,3
4,2018,1 mandato,232.1,15.0%,65.2%,84.0%,2
5,2018,2+ mandatos,305.2,12.0%,63.0%,83.8%,2
6,2022,Sem experiencia,2130.3,12.6%,43.3%,78.4%,3
7,2022,1 mandato,557.0,22.4%,60.2%,88.6%,2
8,2022,2+ mandatos,546.7,27.7%,58.9%,86.8%,2


In [7]:
# Vantagem de experiência por tipo de partido (Competitivo vs. Menos Competitivo)

cadeiras_nacionais = (
    rrd_df_merge[rrd_df_merge["eleito"] == 1]
    .groupby(["ano_eleicao", "sg_partido"]).size()
    .reset_index(name="n_cadeiras")
)
cadeiras_nacionais["tipo_partido"] = np.where(
    cadeiras_nacionais["n_cadeiras"] >= 20, "Competitivo", "Menos competitivo"
)

df_rec_tipo = df_rec_partido.merge(
    cadeiras_nacionais[["ano_eleicao", "sg_partido", "tipo_partido"]],
    on=["ano_eleicao", "sg_partido"], how="left",
)
df_rec_tipo["tipo_partido"] = df_rec_tipo["tipo_partido"].fillna("Menos competitivo")
df_cum_tipo = calc_cumulative(df_rec_tipo, ["ano_eleicao", "tipo_partido", "exp_deputado_federal"])

tipos  = ["Competitivo", "Menos competitivo"]
anos   = [2014, 2018, 2022]

fig3 = make_subplots(
    rows=2, cols=3,
    shared_yaxes=True, shared_xaxes=False,
    row_titles=tipos,
    column_titles=[str(a) for a in anos],
    vertical_spacing=0.10, horizontal_spacing=0.06,
)
for row_idx, tipo in enumerate(tipos, start=1):
    for col_idx, ano in enumerate(anos, start=1):
        d_painel = df_cum_tipo[
            (df_cum_tipo["ano_eleicao"] == ano) & (df_cum_tipo["tipo_partido"] == tipo)
        ]
        for grupo in grupos:
            d = d_painel[d_painel["exp_deputado_federal"] == grupo].sort_values("semana")
            if d.empty:
                continue
            fig3.add_trace(
                go.Scatter(
                    x=d["semana"], y=d["cum_prop"],
                    mode="lines+markers", name=grupo,
                    showlegend=(row_idx == 1 and col_idx == 1),
                    line=dict(color=cores[grupo], dash=tracas[grupo], width=2),
                    marker=dict(size=5),
                ),
                row=row_idx, col=col_idx,
            )

fig3.update_layout(
    title_text="Vantagem de experiência por tipo de partido — proporção acumulada por semana<br>"
               "<sup>† 2014 = FP + PJ_ROTEADO (pré-FEFC); 2018/2022 = FEFC + FP</sup>",
    legend=dict(title="Experiência (Dep. Federal)", orientation="h", y=-0.10, x=0.5, xanchor="center"),
    height=600, width=1100, template="plotly_white",
)
fig3.update_xaxes(title_text="Semana", dtick=1)
fig3.update_yaxes(tickformat=".0%", range=[0, 1.05])
fig3.update_yaxes(title_text="Proporção acumulada", col=1)
fig3.show(config={"staticPlot": True})

## Fluxo Cumulativo por Candidato Competitivo

In [8]:

# Proporção acumulada por semana — candidato competitivo (CS) vs. demais
# Layout 2×2: linhas = tipo de partido (competitivo / menos competitivo), colunas = 2018, 2022
# Candidato ex-ante: incumbente | alcancou_10pct_qe_hist
# Tipo de partido: ≥20 cadeiras pré-eleição = competitivo
# Fonte: receitas de partido (FEFC+FP em 2018/2022)

cs_cols = ["ano_eleicao", "sg_uf", "sg_partido", "nr_candidato", "candidato_competitivo"]
_cs_lookup = rrd_df_merge[cs_cols].drop_duplicates()


def _add_grupo_cs(df_src):
    out = df_src.merge(_cs_lookup, on=["ano_eleicao", "sg_uf", "sg_partido", "nr_candidato"], how="left")
    out["candidato_competitivo"] = out["candidato_competitivo"].fillna(False)
    out["grupo_cs"] = out["candidato_competitivo"].map(
        {True: "Candidato competitivo", False: "Candidato não-competitivo"}
    )
    return out


df_rec_cs         = _add_grupo_cs(df_rec)
df_rec_cs_partido = _add_grupo_cs(df_rec_partido)

# Adicionar tipo de partido (≥20 cadeiras pré-eleição = competitivo)
df_rec_cs_tipo = df_rec_cs_partido.merge(
    cadeiras_nacionais[["ano_eleicao", "sg_partido", "tipo_partido"]],
    on=["ano_eleicao", "sg_partido"], how="left",
)
df_rec_cs_tipo["tipo_partido"] = df_rec_cs_tipo["tipo_partido"].fillna("Menos competitivo")
# Restringir a 2018 e 2022
df_rec_cs_tipo = df_rec_cs_tipo[df_rec_cs_tipo["ano_eleicao"].isin([2018, 2022])].copy()

cores_cs  = {"Candidato competitivo": "black",  "Candidato não-competitivo": "lightgrey"}
tracas_cs = {"Candidato competitivo": "solid",   "Candidato não-competitivo": "dot"}
grupos_cs = ["Candidato competitivo", "Candidato não-competitivo"]
tipos_partido = ["Competitivo", "Menos competitivo"]
anos = [2018, 2022]

df_cum_cs_tipo = calc_cumulative(
    df_rec_cs_tipo, ["ano_eleicao", "tipo_partido", "grupo_cs"]
)

fig_cs = make_subplots(
    rows=2, cols=2,
    row_titles=["Partidos competitivos<br>(≥20 cadeiras pré-eleição)", "Partidos menos competitivos<br>(<20 cadeiras)"],
    column_titles=["2018", "2022"],
    shared_yaxes=True,
    vertical_spacing=0.14, horizontal_spacing=0.06,
)
for row_idx, tipo in enumerate(tipos_partido, start=1):
    for col_idx, ano in enumerate(anos, start=1):
        for grupo in grupos_cs:
            d = df_cum_cs_tipo[
                (df_cum_cs_tipo["ano_eleicao"] == ano) &
                (df_cum_cs_tipo["tipo_partido"] == tipo) &
                (df_cum_cs_tipo["grupo_cs"] == grupo)
            ].sort_values("semana")
            if d.empty:
                continue
            fig_cs.add_trace(
                go.Scatter(
                    x=d["semana"], y=d["cum_prop"],
                    mode="lines+markers", name=grupo,
                    showlegend=(row_idx == 1 and col_idx == 1),
                    line=dict(color=cores_cs[grupo], dash=tracas_cs[grupo], width=2),
                    marker=dict(size=6),
                ),
                row=row_idx, col=col_idx,
            )

fig_cs.update_layout(
    legend=dict(orientation="h", x=0.5, xanchor="center", y=1.10, yanchor="top",
                title="Candidato (ex-ante)"),
    height=720, width=900, template="plotly_white",
    margin=dict(t=100, b=50),
)
fig_cs.update_xaxes(title_text="Semana da campanha", dtick=1)
fig_cs.update_yaxes(tickformat=".0%", range=[0, 1.05])
fig_cs.update_yaxes(title_text="Proporção acumulada", col=1)

fig_cs.write_image("figs/cap3_cumulative_proportion_cs.png", scale=2)
fig_cs.show(config={"staticPlot": True})


C:\Users\yuri.taba\AppData\Local\Temp\ipykernel_23884\3285137008.py:80: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




In [9]:

# Valor absoluto acumulado (R$ milhões) — mesmo layout 2×2 que a proporção

fig_cs_abs = make_subplots(
    rows=2, cols=2,
    row_titles=["Partidos competitivos<br>(≥20 cadeiras pré-eleição)", "Partidos menos competitivos<br>(<20 cadeiras)"],
    column_titles=["2018", "2022"],
    shared_yaxes=False,
    vertical_spacing=0.14, horizontal_spacing=0.12,
)
for row_idx, tipo in enumerate(tipos_partido, start=1):
    for col_idx, ano in enumerate(anos, start=1):
        for grupo in grupos_cs:
            d = df_cum_cs_tipo[
                (df_cum_cs_tipo["ano_eleicao"] == ano) &
                (df_cum_cs_tipo["tipo_partido"] == tipo) &
                (df_cum_cs_tipo["grupo_cs"] == grupo)
            ].sort_values("semana")
            if d.empty:
                continue
            fig_cs_abs.add_trace(
                go.Scatter(
                    x=d["semana"], y=d["cum_receita"] / 1e6,
                    mode="lines+markers", name=grupo,
                    showlegend=(row_idx == 1 and col_idx == 1),
                    line=dict(color=cores_cs[grupo], dash=tracas_cs[grupo], width=2),
                    marker=dict(size=6),
                ),
                row=row_idx, col=col_idx,
            )

fig_cs_abs.update_layout(
    legend=dict(orientation="h", x=0.5, xanchor="center", y=1.10, yanchor="top",
                title="Candidato (ex-ante)"),
    height=720, width=900, template="plotly_white",
    margin=dict(t=100, b=50),
)
fig_cs_abs.update_xaxes(title_text="Semana da campanha", dtick=1)
fig_cs_abs.update_yaxes(title_text="R$ acumulado (milhões)", col=1)

fig_cs_abs.write_image("figs/cap3_cumulative_abs_cs.png", scale=2)
fig_cs_abs.show(config={"staticPlot": True})


C:\Users\yuri.taba\AppData\Local\Temp\ipykernel_23884\412981733.py:40: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




## Composição do Financiamento por Fonte

In [10]:
# Composição do financiamento por fonte — fluxo cumulativo absoluto (R$ mi)
# Revela como a proibição de PJ (2015) e a criação do FEFC (2017) recompuseram o financiamento.

CORES_FONTE = {
    "PJ_DIRETO":              "#e15759",
    "PJ_ROTEADO":             "#f28e2b",
    "FP":                     "#4e79a7",
    "FEFC":                   "#4e79a7",
    "PF":                     "#76b7b2",
    "PROPRIO":                "#59a14f",
    "OUTROS_CAND":            "#b07aa1",
    "FINANCIAMENTO_COLETIVO": "#edc948",
    "OUTROS":                 "#bab0ac",
}
DASHES_FONTE = {
    "PJ_DIRETO": "solid",  "PJ_ROTEADO": "solid", "FP": "dash",
    "FEFC":      "solid",  "PF":         "dot",   "PROPRIO": "dashdot",
    "OUTROS_CAND": "dot",  "FINANCIAMENTO_COLETIVO": "longdash", "OUTROS": "dot",
}
FONTES_ORDEM = {
    2014: ["PJ_DIRETO", "PJ_ROTEADO", "PF", "PROPRIO", "OUTROS_CAND", "FP", "OUTROS"],
    2018: ["FEFC", "PF", "PROPRIO", "OUTROS_CAND", "FP", "FINANCIAMENTO_COLETIVO", "OUTROS"],
    2022: ["FEFC", "PF", "PROPRIO", "OUTROS_CAND", "FP", "FINANCIAMENTO_COLETIVO", "OUTROS"],
}

fig_fontes = make_subplots(
    rows=1, cols=3,
    subplot_titles=["2014 (pré-FEFC)", "2018", "2022"],
    shared_yaxes=False, horizontal_spacing=0.08,
)

seen_legend = set()
for col_idx, ano in enumerate([2014, 2018, 2022], start=1):
    df_ano = df_rec[df_rec["ano_eleicao"] == ano].copy()
    df_cum_f = calc_cumulative(df_ano, ["ano_eleicao", "fonte_tipo"])

    fontes_com_valor = (
        df_ano.groupby("fonte_tipo")["vr_receita"].sum()
        .loc[lambda s: s > 0].index.tolist()
    )
    ordem = [f for f in FONTES_ORDEM[ano] if f in fontes_com_valor]

    for fonte in ordem:
        d = df_cum_f[df_cum_f["fonte_tipo"] == fonte].sort_values("semana")
        if d.empty:
            continue
        total_m = d["vr_receita"].sum() / 1e6
        label = f"{fonte} (R${total_m:.0f}M)"
        show = fonte not in seen_legend
        seen_legend.add(fonte)
        fig_fontes.add_trace(
            go.Scatter(
                x=d["semana"], y=d["cum_receita"] / 1e6,
                mode="lines+markers", name=label,
                legendgroup=fonte, showlegend=show,
                line=dict(color=CORES_FONTE.get(fonte, "#aaaaaa"),
                          dash=DASHES_FONTE.get(fonte, "solid"), width=2.5),
                marker=dict(size=5),
            ),
            row=1, col=col_idx,
        )

fig_fontes.update_layout(
    title_text=(
        "Composição do financiamento de campanha por fonte — Dep. Federal<br>"
        "<sup>PJ_DIRETO e PJ_ROTEADO extintos após STF/2015; FEFC criado em 2017</sup>"
    ),
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.22, yanchor="top",
                font=dict(size=11)),
    height=500, width=1200, template="plotly_white",
    margin=dict(t=70, b=120),
)
fig_fontes.update_xaxes(title_text="Semana da campanha", dtick=1)
fig_fontes.update_yaxes(title_text="R$ acumulado (milhões)")
fig_fontes.show(config={"staticPlot": True})